In [2]:
import yaml
import torch
import pytorch_lightning as pl
from tqdm.auto import tqdm
import torch.nn as nn
import numpy as np 

/om2/user/imgriff/conda_envs/torch_11_cuda_11_pitch/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import sys 
sys.path.append('../')
from src.attn_tracking_lightning import AttentionalTrackingModule
from corpus.jsinV3AttnTrackingValidation import jsinV3_attn_tracking_validation
from corpus.jsinV3_attn_multi_talker_w_audioset import jsinV3_attn_multi_talker_w_audioset
import src.audio_transforms as at


In [ ]:
path = '../config/attentional_cue/attn_cue_lr_1e-4_bs_64_constrained_slope_multi_distractor.yaml'
config = yaml.load(open(path, 'r'), Loader=yaml.FullLoader)

In [ ]:

audio_config = config['data']['audio']

audio_transforms = at.AudioCompose([
            at.AudioToTensor(),
            at.CombineWithRandomDBSNR(low_snr=-10, high_snr=10), # set to same as training 
            at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config),
])

bg_combine_transforms = at.AudioCompose([
        at.AudioToTensor(),
        at.CombineWithRandomDBSNR(low_snr=-10, high_snr=10), # set to same as training 
        at.RMSNormalizeForegroundAndBackground(rms_level=0.1),
        at.UnsqueezeAudio(dim=0),
    ])


cochgram_transforms = at.AudioCompose([
            at.UnsqueezeAudio(dim=0),
            at.AudioToAudioRepresentation(**audio_config),

])

In [ ]:

config['layernorm_first'] = True
config['data']['corpus']['n_talkers'] = 1 

model = AttentionalTrackingModule(config)

dataset = jsinV3_attn_multi_talker_w_audioset(**config['data']['corpus'],
                                          mode='val',
                                              
                                          transform=[audio_transforms,bg_combine_transforms],
                                          demo=True)

dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=0
        )


In [10]:
ckpt_path = '../attn_cue_models/attn_cue_jsin_multi_distractor_w_audioset_bs_64_lr_1e-4/' \
'checkpoints/epoch=0-step=70000.ckpt'
ckpt = torch.load(ckpt_path, map_location=torch.device('cpu'))

model.load_state_dict(ckpt['state_dict'])


<All keys matched successfully>

In [11]:

activations = {}


def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach()
    return hook



In [12]:
n_activations = 100
# set hooks for time_average layers & make SigmoidLayers


conv_modules = {name:module for name, module in model.model.named_children() if 'conv' in name or 'relufc' in name}

fg_reps = {}
bg_reps = {}
mixture_reps = {}

for name, module in conv_modules.items():
    print(name)
    if 'relufc' in name:
        module.register_forward_hook(get_activation(name)) # [2] is relu 
    else:
        module[2].register_forward_hook(get_activation(name)) # [2] is relu 

    fg_reps[name] = []
    bg_reps[name] = []
    mixture_reps[name] = []



conv0
conv1
conv2
conv3
conv4
conv5
conv6
relufc


In [14]:
model = model.eval().cuda()
# model = model.eval()


# add cochleagram to dicts 

mixture_reps['cochleagram'] = []
fg_reps['cochleagram'] = []
bg_reps['cochleagram'] = []


with torch.no_grad():
    for ix, batch in tqdm(enumerate(dataloader),  total = n_activations):
        foreground, background, mixture, fg_cue, fg_target = batch
        
        # convert to cochleagrams
        foreground, _ = cochgram_transforms(foreground, None)
        background, _ = cochgram_transforms(background, None)
        foreeground = foreground.squeeze(0)
        background = background.squeeze(0)


        # save inputs
        mixture_reps['cochleagram'].append(mixture.view(1,-1))
        fg_reps['cochleagram'].append(foreground.view(1,-1))
        bg_reps['cochleagram'].append(background.view(1,-1))

        # send to device
        foreground, background, mixture = foreground.cuda(), background.cuda(), mixture.cuda()
        fg_cue =  fg_cue.cuda()
        
        # run mixture
        model(fg_cue, mixture)
            
        for layer in mixture_reps.keys():
            if layer == 'cochleagram':
                continue 
            mixture_reps[layer].append(activations[layer].view(1,-1).cpu())
                
        # run fg
        model(fg_cue, foreground)
            
        for layer in fg_reps.keys():
            if layer == 'cochleagram':
                continue 
            fg_reps[layer].append(activations[layer].view(1,-1).cpu())
                
        # run bg
        model(fg_cue, background)
            
        for layer in bg_reps.keys():
            if layer == 'cochleagram':
                continue 
            bg_reps[layer].append(activations[layer].view(1,-1).cpu())
        
        if ix == n_activations-1:
            break 
            

 99%|█████████▉| 99/100 [01:14<00:00,  1.33it/s]


In [15]:
mixture_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                     for layer,acts in mixture_reps.items()}

fg_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                for layer,acts in fg_reps.items()}

bg_reps = {layer:torch.concat(acts,axis=0).numpy().astype('float16')
                for layer,acts in bg_reps.items()}


In [16]:
import pickle

out_dict = dict(mixture_reps=mixture_reps, fg_reps=fg_reps, bg_reps=bg_reps)
out_name = '../attn_cue_models/attn_cue_jsin_multi_distractor_w_audioset_bs_64_lr_1e-4/model_output_reps_w_noise.pkl'


with open(out_name ,'wb') as f:
    pickle.dump(out_dict, f, protocol=pickle.HIGHEST_PROTOCOL)
    
out_name = '../attn_cue_models/attn_cue_jsin_multi_distractor_w_audioset_bs_64_lr_1e-4/model_output_reps_w_noise.npz'

np.savez(out_name, mixture_reps=mixture_reps, fg_reps=fg_reps, bg_reps=bg_reps)